# 0. 준비 단계

In [ ]:
!pip install pm4py pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
csv_path = "/content/drive/MyDrive/Colab/mimicel_final.csv"
df = pd.read_csv(csv_path)

# 1. Acuity-based 분석

In [ ]:
# 1) 날짜만 있는 형태를 찾아서 시간 붙이기
df["timestamps"] = df["timestamps"].str.replace(
    r"^(\d{4}-\d{2}-\d{2})$",
    r"\1 00:00:00",
    regex=True
)

# 2) datetime으로 파싱
df["timestamps"] = pd.to_datetime(df["timestamps"], errors="coerce")

# 3) Disco 표준 포맷 문자열로 변환
df["timestamps"] = df["timestamps"].dt.strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# Acuity를 case-level attribute로 전파
acuity_map = df.groupby("stay_id")["acuity"].first()
df["acuity_case"] = df["stay_id"].map(acuity_map)

In [ ]:
# Acuity=3 케이스만 필터링
acu3 = df[df["acuity_case"] == 3]

In [ ]:
# csv로 저장하여 Disco로 넘길 파일 생성
acu3.to_csv("acuity3_only.csv", index=False)

acuity = 1~5인 케이스 필터링

In [ ]:
import pandas as pd
import numpy as np

TARGET_ROWS = 5_000_000   # 목표: 500만 행 (Disco 학생 라이센스 허용 범위)

# ============================================
# 1) 원본 이벤트 로그 로딩
# ============================================
df_1 = pd.read_csv("/content/drive/MyDrive/Colab/mimicel_final.csv", dtype={'activity': str})

# ============================================
# 2) case-level acuity 전파
# ============================================
acuity_map = (
    df_1[df_1['activity'] == 'Triage in the ED'][['stay_id', 'acuity']]
    .drop_duplicates(subset=['stay_id'])
    .set_index('stay_id')['acuity']
)
df_1['acuity_case'] = df_1['stay_id'].map(acuity_map)

# ============================================
# 3) acuity NULL 제거 + 1~5만 남기기
# ============================================
df_1 = df_1[df_1['acuity_case'].notna()]
df_1 = df_1[df_1['acuity_case'].isin([1,2,3,4,5])]

total_rows = len(df_1)
total_cases = df_1['stay_id'].nunique()
print("전체 이벤트 수:", total_rows)
print("전체 stay_id 수:", total_cases)

# ============================================
# 4) 필요한 sampling 비율 계산
# ============================================
required_frac = TARGET_ROWS / total_rows

# 상한선 보정
if required_frac > 1:
    required_frac = 1

print(f"\n500만 행을 위해 필요한 stay_id 샘플링 비율: {required_frac:.4f}")

# ============================================
# 5) stay_id 단위 acuity 정보 준비
# ============================================
case_level = df_1.groupby('stay_id')['acuity_case'].first().reset_index()

# ============================================
# 6) Stratified Sampling (acuity 비율 유지)
# ============================================
sampled_cases = (
    case_level.groupby('acuity_case')
    .sample(frac=required_frac, random_state=42)
)

sampled_ids = sampled_cases['stay_id'].tolist()

print("\n샘플링된 케이스 수:", len(sampled_ids))
print("acuity별 분포:")
print(sampled_cases['acuity_case'].value_counts(normalize=True).sort_index())

# ============================================
# 7) 샘플링된 stay_id 전체 이벤트 추출
# ============================================
df_1_sampled = df_1[df_1['stay_id'].isin(sampled_ids)].copy()

print("\n샘플링 후 이벤트 수:", len(df_1_sampled))
print("샘플링 후 케이스 수:", df_1_sampled['stay_id'].nunique())
print("샘플링 후 acuity 분포:")
print(df_1_sampled['acuity_case'].value_counts(normalize=True).sort_index())

# ============================================
# 8) 저장
# ============================================
# out_path = '/content/mimicel_acuity_stratified.csv'
# df_1_sampled.to_csv(out_path, index=False)

# print(f"\n파일 저장 완료 → {out_path}")

전체 이벤트 수: 7073981
전체 stay_id 수: 418052

500만 행을 위해 필요한 stay_id 샘플링 비율: 0.7068

샘플링된 케이스 수: 295485
acuity별 분포:
acuity_case
1.0    0.057441
2.0    0.333459
3.0    0.538318
4.0    0.068163
5.0    0.002619
Name: proportion, dtype: float64

샘플링 후 이벤트 수: 5001458
샘플링 후 케이스 수: 295485
샘플링 후 acuity 분포:
acuity_case
1.0    0.071808
2.0    0.393015
3.0    0.496577
4.0    0.037422
5.0    0.001177
Name: proportion, dtype: float64


# 2. LoS-driven 분석

In [ ]:
df = df_1_sampled.copy()

# stay_id가 index일 경우 컬럼으로 내려보냄
if df.index.name == 'stay_id':
    df = df.reset_index()

# 중복 컬럼 제거
df = df.loc[:, ~df.columns.duplicated()]

# 확인용
print(df.index.name)
print([c for c in df.columns if c == 'stay_id'])

# ===========================================
# 1) timestamps 파싱
# ===========================================
df['timestamps'] = pd.to_datetime(df['timestamps'], errors='coerce')

# ===========================================
# 2) LoS 계산
# ===========================================
entry = (
    df[df['activity'] == 'Enter the ED']
    .groupby('stay_id')['timestamps']
    .min()
)

exit_ = (
    df[df['activity'] == 'Discharge from the ED']
    .groupby('stay_id')['timestamps']
    .max()
)

los_df = pd.DataFrame({
    'stay_id': entry.index,
    'los_min': (exit_ - entry).dt.total_seconds() / 60
})

los_df = los_df.reset_index(drop=True)
los_df = los_df.loc[:, ~los_df.columns.duplicated()]

# LoS 그룹
los_df['los_group'] = np.where(
    los_df['los_min'] <= 500, 'Normal', 'Prolonged'
)

# ===========================================
# 3) Acuity high/low 설정
# ===========================================
df['acuity_group'] = df['acuity_case'].map(
    lambda x: 'High' if x in [1, 2] else 'Low'
)

# ===========================================
# ⭐ 4) Disposition case-level 전파 ⭐
# (Discharge 이벤트에서만 disposition이 정확하므로 그 값을 stay_id별로 전달)
# ===========================================
disp_map = (
    df[df['activity'] == 'Discharge from the ED'][['stay_id', 'disposition']]
    .drop_duplicates(subset=['stay_id'])
    .set_index('stay_id')['disposition']
)

df['disposition_case'] = df['stay_id'].map(disp_map)

# ===========================================
# 5) df + LoS 병합
# ===========================================
df = df.merge(los_df, on='stay_id', how='left')

# ===========================================
# 6) Q1~Q4 zone 분류
# ===========================================
def zone_fn(row):
    if row['acuity_group'] == 'High' and row['los_group'] == 'Normal':
        return 'Q1'
    if row['acuity_group'] == 'Low' and row['los_group'] == 'Normal':
        return 'Q2'
    if row['acuity_group'] == 'Low' and row['los_group'] == 'Prolonged':
        return 'Q3'
    if row['acuity_group'] == 'High' and row['los_group'] == 'Prolonged':
        return 'Q4'
    return None

df['zone'] = df.apply(zone_fn, axis=1)

# ===========================================
# 7) 결과 점검
# ===========================================
print("전체 케이스:", df['stay_id'].nunique())

print("\nLoS group 분포:")
print(df.drop_duplicates('stay_id')['los_group'].value_counts())

print("\nzone 분포 (케이스 기준):")
print(df.drop_duplicates('stay_id')['zone'].value_counts(normalize=True))

print("\nDisposition 분포:")
print(df.drop_duplicates('stay_id')['disposition_case'].value_counts())

None
['stay_id']
전체 케이스: 295485

LoS group 분포:
los_group
Normal       221405
Prolonged     74080
Name: count, dtype: int64

zone 분포 (케이스 기준):
zone
Q2    0.477226
Q1    0.272068
Q3    0.131875
Q4    0.118832
Name: proportion, dtype: float64

Disposition 분포:
disposition_case
HOME                           169975
ADMITTED                       108431
TRANSFER                         4883
LEFT WITHOUT BEING SEEN          4269
ELOPED                           4013
OTHER                            2438
LEFT AGAINST MEDICAL ADVICE      1341
EXPIRED                           135
Name: count, dtype: int64


In [ ]:
out_path = "/content/mimicel_los_zone.csv"
df.to_csv(out_path, index=False)
print(f"\n 최종 LoS-driven 파일 저장 완료 → {out_path}")


 최종 LoS-driven 파일 저장 완료 → /content/mimicel_los_zone.csv


# 3. Crowdedness 분석

In [ ]:
import pandas as pd
import numpy as np

# =============================
# 1) 각 stay_id별 enter/discharge 시간 추출
# =============================

enter_time = (
    df[df['activity'] == 'Enter the ED']
    .groupby('stay_id')['timestamps']
    .min()
)

exit_time = (
    df[df['activity'] == 'Discharge from the ED']
    .groupby('stay_id')['timestamps']
    .max()
)

intervals = pd.DataFrame({
    'stay_id': enter_time.index,
    'enter': enter_time.values,
    'exit': exit_time.values
})

# =============================
# 2) 효율적 concurrency 계산을 위한 이벤트 타임라인 생성
# =============================
events = []

for sid, start, end in intervals[['stay_id','enter','exit']].itertuples(index=False):
    # 환자 입실 시 +1
    events.append((start, 1))
    # 환자 퇴실 시 -1
    events.append((end, -1))

# timestamp 기준 정렬
events.sort(key=lambda x: x[0])

# =============================
# 3) 스윕라인 알고리즘으로 현재 ED 환자 수 계산
# =============================
concurrency = {}   # stay_id -> concurrency count 저장
current = 0

# 각 시간에서 ED에 머물고 있는 환자 수를 기록
time_points = []

for (t, delta) in events:
    current += delta
    time_points.append((t, current))

# =============================
# 4) 각 stay_id 체류 중 최대 concurrency 계산
# =============================
from bisect import bisect_left, bisect_right

# 이벤트들의 시간/동시환자수 배열로 분리
times = [tp[0] for tp in time_points]
vals = [tp[1] for tp in time_points]

def get_max_concurrency(start, end):
    """ 주어진 [start, end] 기간 동안의 최대 동시환자 수 """
    # start~end 구간 해당 index 범위 찾기
    l = bisect_left(times, start)
    r = bisect_right(times, end)
    if l == len(times):
        return 0
    return max(vals[l:r]) if l<r else 0

intervals['concurrency'] = intervals.apply(
    lambda row: get_max_concurrency(row['enter'], row['exit']),
    axis=1
)

# =============================
# 5) 75 percentile → crowded threshold
# =============================
threshold = np.percentile(intervals['concurrency'], 75)
print("혼잡도 threshold (75 percentile):", threshold)

# 논문 값과 동일해야 함 → 12
threshold = int(threshold)

# =============================
# 6) crowded 여부 부여
# =============================
intervals['crowded_flag'] = intervals['concurrency'] >= threshold

# =============================
# 7) df에 case-level crowded_flag 전파
# =============================
df = df.merge(intervals[['stay_id','concurrency','crowded_flag']], on='stay_id', how='left')

print(df[['stay_id','concurrency','crowded_flag']].head())

혼잡도 threshold (75 percentile): 7.0
    stay_id  concurrency  crowded_flag
0  30000012            7          True
1  30000012            7          True
2  30000012            7          True
3  30000012            7          True
4  30000012            7          True


In [ ]:
df.to_csv("mimicel_crowded_final.csv", index=False)